#### Reference: minBPE Tokenization lecture by Andrej Karpathy (https://youtu.be/zduSFxRajkE?si=CxVSuSDkOlN-g8rl)

In [1]:
# raw text
text = open("tokenizer_data/unicode_blogpost.txt", "r").read()
# text = open("tokenizer_data/taylor_swift.txt", "r").read()

# convert text to utf-8 encoding bytes
raw_bytes = text.encode("utf-8")

# convert bytes to their integer/decimal representation --> base vocab = int from 0 to 255
token_ids = list(raw_bytes)  # often verbosely written as list(map(int, raw_bytes))

print(f"text length = {len(text)} | raw bytes = {len(raw_bytes)} | total tokens = {len(token_ids)}")

text length = 23551 | raw bytes = 24827 | total tokens = 24827


#### Building individual components

In [ ]:
# get all the pairs of tokens
def get_pair_frequency(ids):
    counts = {}
    for id0, id1 in zip(ids, ids[1:]):
        counts[(id0, id1)] = 1 + counts.get((id0, id1), 0)
    return counts


# test the function
stats = get_pair_frequency(ids=token_ids)
print(stats)

In [ ]:
# get most frequently occurring pair of token IDs
pair = max(stats, key=lambda x: stats[x])

print(f"Pair {pair} occurs most frequently -- {stats[pair]} times")

In [ ]:
# merge the pair with a new token
# iterates through the entire sequence and replaces the "pair" of token ids with a newly minted token


def merge(ids, pair, new_token):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


# test function
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

#### Full BPE Training

In [2]:
# BPE training


def get_pair_frequency(ids):
    counts = {}
    for id0, id1 in zip(ids, ids[1:]):
        counts[(id0, id1)] = 1 + counts.get((id0, id1), 0)
    return counts


def merge(ids, pair, new_token):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


# parameters
num_merges = 20
vocab_size = 256 + num_merges

# data
ids = token_ids.copy()

# model parameters
merges = {}  # maps merged pairs to new token

for i in range(num_merges):
    # get most frequently occurring token pair
    stats = get_pair_frequency(ids)
    top_pair = max(stats, key=lambda x: stats[x])
    # mint new token and store the mapping
    new_token = 255 + i
    merges[top_pair] = new_token
    print(f"Merging {top_pair} into a new token {new_token}")
    # replace pair with new token
    ids = merge(ids=ids, pair=top_pair, new_token=new_token)


Merging (101, 32) into a new token 255
Merging (105, 110) into a new token 256
Merging (115, 32) into a new token 257
Merging (116, 104) into a new token 258
Merging (101, 114) into a new token 259
Merging (99, 111) into a new token 260
Merging (116, 32) into a new token 261
Merging (226, 128) into a new token 262
Merging (44, 32) into a new token 263
Merging (97, 110) into a new token 264
Merging (111, 114) into a new token 265
Merging (100, 32) into a new token 266
Merging (97, 114) into a new token 267
Merging (101, 110) into a new token 268
Merging (260, 100) into a new token 269
Merging (256, 103) into a new token 270
Merging (121, 32) into a new token 271
Merging (97, 108) into a new token 272
Merging (111, 110) into a new token 273
Merging (258, 255) into a new token 274


In [3]:
print(f"Original vocab size = 256 and sequence length = {len(token_ids)}")
print(f"Original vocab size = {vocab_size} and sequence length = {len(ids)}")
print(f"Compression ratio: {len(token_ids) / len(ids):.2f}X")

Original vocab size = 256 and sequence length = 24827
Original vocab size = 276 and sequence length = 19668
Compression ratio: 1.26X
